# Single-qubit autocalibration (co-simulation)

The co-sim analogue of the hardware `single_qubit_autocalibrate_v2` notebook. It drives the
[`autocal.py`](autocal.py) script — plain host python, no kernel DSL — which runs

> spectroscopy → Ramsey (frequency) → Rabi (amplitude) → T1

gated by an `expts_to_run` dict, with **incremental frequency updates** and the automatic
**Ramsey-after-Amplitude re-run** rule (an amplitude change moves the AC-Stark-shifted qubit
frequency, so the frequency calibration is re-run). Same `TwoLevelModel` ground truth as the
single-qubit calibration notebook.

In [1]:
import os, sys, math
from pathlib import Path

import numpy as np

import riscq
from riscq.cal import Config
from riscq.cal.base import bind_params, gate_sigma, GATE_ENV
from riscq.map import SocMap, SocParams
from riscq.pulses import Pulse, units
from riscq.sim import server

sys.path.insert(0, os.getcwd())          # the autocal.py script sits beside this notebook
from autocal import single_qubit_autocalibrate

SW = Path(riscq.__file__).resolve().parents[1]
F_GE = 50e6

## Bring up the co-sim, plant the ground truth, detune the Config

In [2]:
drv = server.start(SW / 'configs' / 'sim-2q.json', SW / 'build' / 'sim-2q')
m = SocMap(SocParams.from_json(drv.sim.get_params()))
bind_params(m)

sig_max = gate_sigma(m, Pulse(GATE_ENV, freq_hz=F_GE, amp=0.5), F_GE, units.AMP_SCALE - 600)
rabi = float(3 * math.pi / sig_max)
drv.sim.set_model(dict(kind='twolevel', core=0, rabi_rad_per_amp=rabi, readout_code=2048,
                       readout_amp=20000.0, readout_phase=0.0, f_ge=F_GE,
                       t1=200, t2=350, noise_scale=120.0, noise_seed=1))
g = sig_max / (units.AMP_SCALE - 600)
true_x90_amp = (math.pi / 2) / (rabi * g) / units.AMP_SCALE

# Detune BOTH the readout (spectroscopy fixes it first) and the qubit drive frequency.
d0_code = 60
drive = units.code_to_freq(units.freq_to_code(F_GE, m.params) + d0_code, m.params)
cfg = Config()
cfg['qubit/0/freq'] = float(drive)
cfg['qubit/0/x90/amp'] = 0.5
cfg['qubit/0/T1'] = 200
cfg['readout/0/freq'] = float(units.demod_code_to_freq(2048 + 100, m.params))  # readout off by 100 codes
cfg['readout/0/dur'] = 40
print(f'start: qubit freq err={abs(drive - F_GE):.3g} Hz, X90 amp=0.5 (true≈{true_x90_amp:.4f})')

start: qubit freq err=1.46e+06 Hz, X90 amp=0.5 (true≈0.1616)


## Run the autocalibration

All four experiments enabled. `single_qubit_autocalibrate` applies each successful fit into
`cfg`; because the amplitude moves a lot (0.5 → ≈0.16), the AC-Stark rule fires an extra
`ramsey_restark` step after Rabi.

In [3]:
expts_to_run = {'spectroscopy': True, 'ramsey': True, 'rabi': True, 't1': True}

results = single_qubit_autocalibrate(
    cfg, 0, drv, expts_to_run=expts_to_run, verbose=True,
    spec_kw={'points': 17},
    ramsey_kw={'detune_code': 200, 'n_detune': 4, 'points': 14, 'dt': 4},
    rabi_kw={'n_gates': 1, 'points': 9},
    t1_kw={'points': 8})

print('\nsteps run:', list(results))

  spectroscopy: ok=True proposal={'readout/0/freq': 12500000.0}


/config/build/agentic-rv-dev/software/riscq/cal/fits.py:66: OptimizeWarning: Covariance of the parameters could not be estimated
  popt, pcov = curve_fit(model, x, y, p0=p0, maxfev=20000)


  ramsey: ok=True proposal={'qubit/0/freq': 50000000.0}


  rabi: ok=True proposal={'qubit/0/x90/amp': 0.1614019986391616, 'qubit/0/rabi': 7.427164986516024e-05}
  amplitude moved 0.5000 -> 0.1614; re-running Ramsey (AC-Stark)


  ramsey_restark: ok=True proposal={'qubit/0/freq': 50000000.0}


  t1: ok=True proposal={'qubit/0/T1': 199.73050823076875}

steps run: ['spectroscopy', 'ramsey', 'rabi', 'ramsey_restark', 't1']


## Results

In [4]:
print('readout freq :', round(cfg['readout/0/freq'], 3), 'Hz  (spectroscopy)')
print(f"qubit  freq  : err {abs(drive - F_GE):.3g} -> {abs(cfg['qubit/0/freq'] - F_GE):.3g} Hz")
print(f"X90 amp      : 0.5 -> {cfg['qubit/0/x90/amp']:.4f}  (true≈{true_x90_amp:.4f})")
if 't1' in results and results['t1'].ok:
    print(f"T1           : {cfg['qubit/0/T1']:.1f} batches  (planted 200)")
print('AC-Stark re-run present:', 'ramsey_restark' in results)
assert 'ramsey_restark' in results, 'the amplitude changed, so Ramsey should have re-run'

readout freq : 12500000.0 Hz  (spectroscopy)
qubit  freq  : err 1.46e+06 -> 0 Hz
X90 amp      : 0.5 -> 0.1614  (true≈0.1616)
T1           : 199.7 batches  (planted 200)
AC-Stark re-run present: True


## Gating: `expts_to_run` skips experiments

A fast re-check that runs **only** the readout spectroscopy — the qubit experiments (Ramsey,
Rabi, T1) are gated off, so nothing else runs and no `ramsey_restark` fires. `expts_to_run`
is how the autocal routine turns individual experiments on and off.

In [5]:
only_spec = single_qubit_autocalibrate(
    cfg, 0, drv, expts_to_run={'ramsey': False, 'rabi': False, 't1': False},
    verbose=True, spec_kw={'points': 17})
print('\nsteps run:', list(only_spec))
assert list(only_spec) == ['spectroscopy'], 'gating should have run only spectroscopy'

  spectroscopy: ok=True proposal={'readout/0/freq': 12500000.0}

steps run: ['spectroscopy']


## Tear down the co-sim

In [6]:
drv.sim.set_model({'kind': 'zero'})
server.stop(drv)
print('cosim stopped')

cosim stopped
